In [1]:
import pandas as pd

In [2]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [3]:
import seaborn as sns

In [4]:
## create dict sites plots 

merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt',nrows=1, sep = '\t')

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.T

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index()

merged_hapFIRE_allele_frequency['site'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[0]
merged_hapFIRE_allele_frequency['gen'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[1]
merged_hapFIRE_allele_frequency['plot'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[2]

samples = merged_hapFIRE_allele_frequency.drop([0, 'index'],axis=1).drop_duplicates()

samples_at_least_2_years = samples[['site', 'plot']].value_counts()[samples[['site', 'plot']].value_counts() > 1 ].reset_index()

sites_plots_w_at_least_2_years =  samples_at_least_2_years[['site', 'plot']].groupby('site')['plot'].unique().to_dict()

In [5]:
## test 

In [6]:
flowers = pd.read_csv('../key_files/merged_sample_table.csv')

var_pos = pd.read_csv('../key_files/var_pos_grenenet.csv')
#path_meixi = '/carnegie/data/Shared/Labs/Moi/Everyone/meixilin'

num_flowers_map = flowers.set_index('sample_name')['total_flower_counts'].to_dict()

merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt',nrows=1, sep = '\t')

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.T

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index()

merged_hapFIRE_allele_frequency['site'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[0]
merged_hapFIRE_allele_frequency['gen'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[1]
merged_hapFIRE_allele_frequency['plot'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[2]


#mask = var_pos['maf05filter'].notna()

In [7]:
site = '1'

In [8]:
site_plot_samples = [i for i in merged_hapFIRE_allele_frequency['index'] if i.startswith(str(site) + '_')]

In [9]:
merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt',nrows=500, sep = '\t', usecols = site_plot_samples)

#merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency[mask]

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index(drop=True)



## generate the allale counts
num_flowers_map = flowers.set_index('sample_name')['total_flower_counts'].to_dict()

allele_counts = {}
for i in merged_hapFIRE_allele_frequency.columns:
    num_flowers = num_flowers_map[i]
    allele_counts[i] = merged_hapFIRE_allele_frequency[i] * num_flowers * 2

allele_counts = pd.concat(allele_counts,axis=1)

allele_counts = allele_counts.T

allele_counts = allele_counts.reset_index()


allele_counts['gen'] = allele_counts['index'].str.split('_').str[1].astype(int)
allele_counts['plot'] = allele_counts['index'].str.split('_').str[2].astype(int)


allele_counts = allele_counts.drop('index',axis=1)

allele_counts = allele_counts.melt(id_vars=['gen', 'plot'])

allele_counts.columns = ['gen', 'plot', 'snp', 'count']

allele_counts = allele_counts.drop('plot',axis=1)

In [10]:
allele_counts

,gen,snp,count
0,1,0,13.293954
1,1,0,8.172854
2,1,0,20.999298
3,1,0,24.278624
4,1,0,24.129688
...,...,...,...
16995,3,499,0.404056
16996,3,499,1.897332
16997,3,499,0.017204
16998,3,499,28.722642


In [11]:
merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt',nrows=1, sep = '\t')

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.T

merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index()

merged_hapFIRE_allele_frequency['site'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[0]
merged_hapFIRE_allele_frequency['gen'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[1]
merged_hapFIRE_allele_frequency['plot'] = merged_hapFIRE_allele_frequency['index'].str.split('_').str[2]

In [12]:
unique_sites = merged_hapFIRE_allele_frequency['site'].unique()

In [13]:
pwd -P

'/carnegie/nobackup/scratch/tbellagio/gea_grene-net/snp_origin'

In [14]:
import random
import subprocess

In [59]:
# create sbatch files to submit on cedar server
shfiles = []
for site in sites_plots_w_at_least_2_years:
    seed = random.randint(1,100000000)
    file = f'changep_time{site}.sh'
    cmd = f'python calc_delta_time_allplots_binreg.py {site}'
    text = f'''#!/bin/bash
#SBATCH --job-name=changep_time_{site}
#SBATCH --time=24:00:00  # 
#SBATCH --ntasks=1
#SBATCH --mem-per-cpu=30gb
#SBATCH --output=changep_time_{site}_%j.out
#SBATCH --error=changep_time_{site}_%j.err
#SBATCH --mail-user=tbellagio@carnegiescience.edu
#SBATCH --mail-type=FAIL

source /home/tbellagio/miniforge3/etc/profile.d/conda.sh
conda activate /home/tbellagio/miniforge3/envs/pipeline_snakemake
cd /carnegie/nobackup/scratch/tbellagio/gea_grene-net/snp_origin
{cmd}

'''
    with open(file, 'w') as o:
        o.write("%s" % text)
    shfiles.append(file)

In [60]:
subprocess.run(["sbatch", shfiles[0]], check=True)

Submitted batch job 31624


CompletedProcess(args=['sbatch', 'changep_time1.sh'], returncode=0)

In [61]:
## now run the shfiles
for shfile in shfiles[1:]:
    # Submit each sbatch script to the SLURM scheduler
    subprocess.run(["sbatch", shfile], check=True)

Submitted batch job 31626
Submitted batch job 31627
Submitted batch job 31628
Submitted batch job 31629
Submitted batch job 31630
Submitted batch job 31631
Submitted batch job 31632
Submitted batch job 31633
Submitted batch job 31634
Submitted batch job 31635
Submitted batch job 31636
Submitted batch job 31637
Submitted batch job 31638
Submitted batch job 31639
Submitted batch job 31640
Submitted batch job 31641
Submitted batch job 31642
Submitted batch job 31643
Submitted batch job 31644
Submitted batch job 31645
Submitted batch job 31646
Submitted batch job 31647
Submitted batch job 31648
Submitted batch job 31649
Submitted batch job 31650


In [ ]:
import pandas as pd
import statsmodels.api as sm
import json
from statsmodels.formula.api import ols
import argparse
import logging

# Set up logging

# Set up the argument parser
parser = argparse.ArgumentParser(description='Process some site and plot identifiers.')

# Add arguments to the parser
parser.add_argument('site', type=str, help='The site identifier to process')

# Parse the command line arguments
args = parser.parse_args()

# Extract the site and plot values from the arguments
site = args.site
print(site)


logging.basicConfig(filename=f'log_site_{site}.log', level=logging.DEBUG)

logging.debug(f'Starting script for site: {site}')

In [15]:
site = 24

In [16]:




flowers = pd.read_csv('../key_files/merged_sample_table.csv')
num_flowers_map = flowers.set_index('sample_name')['total_flower_counts'].to_dict()
## ge thte mask for the 0.05maf filtering 
var_pos = pd.read_csv('../key_files/var_pos_grenenet.csv')
mask = var_pos['maf05filter'].notna()


## get the samples i need 
merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt',nrows=1, sep = '\t')
merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.T
merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index()
site_plot_samples = [i for i in merged_hapFIRE_allele_frequency['index'] if i.startswith(str(site) + '_')]
print(site_plot_samples)
## import the dataset 

merged_hapFIRE_allele_frequency = pd.read_csv('../key_files/merged_hapFIRE_allele_frequency.txt', sep = '\t', usecols = site_plot_samples, nrows = 1000)
## filter


merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency[mask]
# reset index after filtering 
merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency.reset_index(drop=True)


## generate the allale counts
num_flowers_map = flowers.set_index('sample_name')['total_flower_counts'].to_dict()




allele_counts_alt = {}
allele_counts_ref = {}
for i in merged_hapFIRE_allele_frequency.columns:
    num_flowers = num_flowers_map[i]
    allele_counts_alt[i] = merged_hapFIRE_allele_frequency[i] * num_flowers * 2
    maj = 1 - merged_hapFIRE_allele_frequency[i]
    allele_counts_ref[i] = maj * num_flowers * 2

allele_counts_alt = pd.concat(allele_counts_alt,axis=1)
allele_counts_alt = allele_counts_alt.T
allele_counts_alt = allele_counts_alt.reset_index()


allele_counts_ref = pd.concat(allele_counts_ref,axis=1)
allele_counts_ref = allele_counts_ref.T
allele_counts_ref = allele_counts_ref.reset_index()


allele_counts_alt['gen'] = allele_counts_alt['index'].str.split('_').str[1].astype(int)
allele_counts_alt['plot'] = allele_counts_alt['index'].str.split('_').str[2].astype(int)
allele_counts_alt = allele_counts_alt.drop('index',axis=1)
allele_counts_alt = allele_counts_alt.melt(id_vars=['gen', 'plot'])
allele_counts_alt.columns = ['gen', 'plot', 'snp', 'count_alt']
allele_counts_alt = allele_counts_alt.drop('plot',axis=1)

allele_counts_ref['gen'] = allele_counts_ref['index'].str.split('_').str[1].astype(int)
allele_counts_ref['plot'] = allele_counts_ref['index'].str.split('_').str[2].astype(int)
allele_counts_ref = allele_counts_ref.drop('index',axis=1)
allele_counts_ref = allele_counts_ref.melt(id_vars=['gen', 'plot'])
allele_counts_ref.columns = ['gen', 'plot', 'snp', 'count_ref']
allele_counts_ref = allele_counts_ref.drop('plot',axis=1)

['24_1_1', '24_1_2', '24_1_3', '24_1_4', '24_1_5', '24_1_7', '24_1_9', '24_1_10', '24_1_11', '24_1_12', '24_2_10', '24_2_11']


/tmp/ipykernel_1807290/3507412920.py:20: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  merged_hapFIRE_allele_frequency = merged_hapFIRE_allele_frequency[mask]


In [17]:
allele_counts = pd.concat([allele_counts_alt,allele_counts_ref['count_ref']],axis=1)

In [18]:
allele_counts['total'] = allele_counts['count_alt'] + allele_counts['count_ref']

In [19]:
allele_counts['gen'].unique()

array([1, 2])

In [20]:
allele_counts[allele_counts['gen']==2]

,gen,snp,count_alt,count_ref,total
10,2,0,3.148948,10.851052,14.0
11,2,0,0.109414,7.890586,8.0
22,2,1,3.973592,10.026408,14.0
23,2,1,0.045216,7.954784,8.0
34,2,2,6.753580,7.246420,14.0
...,...,...,...,...,...
3059,2,254,0.062672,7.937328,8.0
3070,2,255,0.020086,13.979914,14.0
3071,2,255,0.062672,7.937328,8.0
3082,2,256,0.089526,13.910474,14.0


In [21]:
allele_counts[allele_counts['snp'] == 0].to_csv('one_snp.csv')

In [22]:
import json

In [23]:
allele_counts['gen'].unique()

array([1, 2])

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
gen_variable_scaled = scaler.fit_transform(group['gen'].values.reshape(-1, 1))

NameError: name 'group' is not defined

In [25]:
group['gen'].values.reshape(-1, 1)

NameError: name 'group' is not defined

In [26]:
gen_variable_scaled

NameError: name 'gen_variable_scaled' is not defined

In [27]:
coef = {}
for snp, group in allele_counts.groupby('snp'):
    if len(group) > 1:  # Ensure there's enough data for regression



        gen = group['gen']
        
        successes = group.loc[:,'count_alt']
        failures = group.loc[:,'count_ref']
        # Set up the binomial regression model
        
        
        X = sm.add_constant(gen)  # Adding constant for intercept
        y = pd.concat([successes,failures],axis=1)
        # Fit the modelb
        model = sm.GLM(y, X, family=sm.families.Binomial())
        result = model.fit()
        
        # Extract slope (coefficient for environmental variable) and p-value
        slope = result.params[1].item()  # Coefficient for the environmental variable
        p_value = result.pvalues[1].item()   # P-value for the environmental variable
        
        # Prepare the result as a JSON object
        result = {
            "snp": str(snp),
            "slope": slope,
            "p_value": p_value,
        }
        
        # Append result to buffer
        coef[snp] = [slope, p_value]
        

In [28]:
pd.DataFrame(coef).T

,0,1
0,-0.969218,0.109431
1,1.408018,0.014575
2,1.361265,0.004323
3,-1.847777,0.700101
4,-1.847777,0.700101
...,...,...
252,-3.386851,0.417880
253,-3.482402,0.378832
254,-3.706602,0.287425
255,-3.706602,0.287425


In [27]:
pd.DataFrame(coef).T

,0,1
0,-0.969218,0.109431
1,1.408018,0.014575
2,1.361265,0.004323
3,-1.847777,0.700101
4,-1.847777,0.700101
...,...,...
252,-3.386851,0.417880
253,-3.482402,0.378832
254,-3.706602,0.287425
255,-3.706602,0.287425


In [54]:


# Open the file in append mode to add to existing allele_counts without overwriting
with open(f'results_site_{site}_br_test.jsonl', 'a') as file:
    buffer = []
    for snp, group in allele_counts.groupby('snp'):
        if len(group) > 1:  # Ensure there's enough data for regression
            
            gen = group['gen']
            
            successes = group.loc[:,'count_alt']
            failures = group.loc[:,'count_ref']
            # Set up the binomial regression model
            X = sm.add_constant(gen)  # Adding constant for intercept
            y = pd.concat([successes,failures],axis=1)
            # Fit the modelb
            model = sm.GLM(y, X, family=sm.families.Binomial())
            result = model.fit()
            
            # Extract slope (coefficient for environmental variable) and p-value
            slope = result.params[1].item()  # Coefficient for the environmental variable
            p_value = result.pvalues[1].item()   # P-value for the environmental variable
            
            # Prepare the result as a JSON object
            result = {
                "snp": str(snp),
                "slope": slope,
                "p_value": p_value,
            }
            
            # Append result to buffer
            buffer.append(result)
            
            # Check if buffer has reached the chunk size of 100
            if len(buffer) == 100:
                # Write all buffered items to the file as JSON Lines
                for item in buffer:
                    file.write(json.dumps(item) + '\n')
                # Clear the buffer
                buffer.clear()
    # Write all results to the file as JSON Lines
    if buffer:
        for item in buffer:
            file.write(json.dumps(item) + '\n')

In [ ]:
# from the stats model documentation 

#endog for Binomial can be specified in one of three ways: A 1d array of 0 or 1 values, indicating failure or success respectively. 
#A 2d array, with two columns. The first column represents the success count and the second column represents the failure count. A 
#    1d array of proportions, indicating the proportion of successes, with parameter var_weights containing the number of trials for each row.

In [ ]:
import statsmodels.api as sm
import pandas as pd
import json

# Open the file in append mode to add to existing allele_counts without overwriting
with open(f'results_site_{site}_br_test_alt.jsonl', 'a') as file:
    buffer = []
    
    for snp, group in allele_counts.groupby('snp'):
        if len(group) > 1:  # Ensure there's enough data for regression
            
            gen = group['gen']
            successes = group['count_alt']
            total = group['count_alt'] + group['count_ref']  # Total alleles (trials)
            
            # Set up the binomial regression model
            X = sm.add_constant(gen)  # Adding constant for intercept
            model = sm.GLM(successes, X, family=sm.families.Binomial(), var_weights=total)
            
            result = model.fit()
            
            # Extract slope (coefficient for environmental variable) and p-value
            slope = result.params[1].item()  # Coefficient for the environmental variable
            p_value = result.pvalues[1].item()   # P-value for the environmental variable
            
            # Prepare the result as a JSON object
            result_dict = {
                "snp": str(snp),
                "slope": slope,
                "p_value": p_value,
            }
            
            # Append result to buffer
            buffer.append(result_dict)
            
            # Check if buffer has reached the chunk size of 100
            if len(buffer) == 100:
                # Write all buffered items to the file as JSON Lines
                for item in buffer:
                    file.write(json.dumps(item) + '\n')
                # Clear the buffer
                buffer.clear()
    
    # Write any remaining results to the file
    if buffer:
        for item in buffer:
            file.write(json.dumps(item) + '\n')


In [56]:

results = []
with open(f'results_site_{site}_br_test.jsonl', 'r') as file:
    for line in file:
        # Convert JSON string to Python dictionary
        data = json.loads(line)
        # Append the dictionary to the list
        results.append(data)

# Convert the list of dictionaries to a pandas DataFrame
df = pd.DataFrame(results)

In [ ]:
_br_test_alt

In [73]:

results = []
with open(f'results_site_{site}_br_test_alt.jsonl', 'r') as file:
    for line in file:
        # Convert JSON string to Python dictionary
        data = json.loads(line)
        # Append the dictionary to the list
        results.append(data)

# Convert the list of dictionaries to a pandas DataFrame
df = pd.DataFrame(results)

In [74]:
df

,snp,slope,p_value
0,0,-3.413928e+17,0.0
1,1,-1.484185e+17,0.0
2,2,-1.593364e+17,0.0
3,3,-4.860327e+16,0.0
4,4,-4.860327e+16,0.0
...,...,...,...
252,252,-6.489661e+16,0.0
253,253,-2.163648e+17,0.0
254,254,-5.265843e+16,0.0
255,255,-5.265843e+16,0.0
